# Indexing

In [1]:
from qdrant_client import QdrantClient
from langchain_qdrant import QdrantVectorStore
from qdrant_client.models import Distance, VectorParams
from mlx_embedder import MLXQwenEmbeddings
from docling_parser import DoclingParserWrapper


# 1. Initialize your local MLX embedding model
qwen_embeddings = MLXQwenEmbeddings(model_id = "mlx-community/all-MiniLM-L6-v2-4bit")

/Users/chupeng/Documents/Meng_CCDS/insrance_consult/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 12 files: 100%|████████████████████████████████████████████████████████████| 12/12 [00:00<00:00, 87992.39it/s]


In [3]:
# 2. Connect to your local Qdrant instance
client = QdrantClient(url="http://localhost:6333")
collection_name = "life_insurance_policies"

# 3. Bind the MLX embeddings to the Vector Store
vector_store = QdrantVectorStore(
    client=client,
    collection_name=collection_name,
    embedding=qwen_embeddings
)

In [2]:
if client.collection_exists(collection_name):
    client.delete_collection(collection_name)
    print(f"Deleted old collection: {collection_name}")

client.create_collection(
    collection_name=collection_name,
    vectors_config=VectorParams(
        size=384,
        distance=Distance.COSINE
    )
)
print(f"Created new collection: {collection_name}")

Deleted old collection: life_insurance_policies
Created new collection: life_insurance_policies


In [4]:
docling_parser = DoclingParserWrapper()

In [5]:
from indexer import PolicyIndexer

indexer = PolicyIndexer(
    vector_store=vector_store,
    parser=docling_parser 
)

In [6]:
indexer.process_pdfs('../raw_policies/aia')

Processing PDFs in ../raw_policies/aia
Listing files: ['AIA Pro Lifetime Protector (II).pdf', 'AIA Guaranteed Protect Plus (III).pdf']
Processing file: AIA Pro Lifetime Protector (II).pdf
Chunked AIA Pro Lifetime Protector (II).pdf into 76 chunks.
Indexed AIA Pro Lifetime Protector (II).pdf with 76 chunks.
Processing file: AIA Guaranteed Protect Plus (III).pdf
Chunked AIA Guaranteed Protect Plus (III).pdf into 147 chunks.
Indexed AIA Guaranteed Protect Plus (III).pdf with 147 chunks.


# Retrieving

In [4]:
from retriever import PolicyRetriever

In [5]:
retriever = PolicyRetriever(vector_store=vector_store)

In [6]:
test_query = "What are the terms for early stage critical illness?"

In [9]:
results = retriever.retrieve(
    query=test_query, 
    top_k=5
)

In [10]:
for i, chunk in enumerate(results):
    print(f"--- Chunk {i+1} ---")
    print(chunk)
    print("\n")

--- Chunk 1 ---
## Critical Illnesses and Definition  
<!-- image -->


--- Chunk 2 ---
## Critical Illnesses and Definition


--- Chunk 3 ---
## Critical Illnesses and Definition  
<!-- image -->  
## Critical Illnesses and Definition  
The Diagnosis must be based on all of the following criteria:  
- hyper-gammaglobulinaemia
- the presence of at least one of the following auto-antibodies:
- -Anti-Nuclear Antibody;
- -Anti-smooth muscle antibodies;
- -Anti-actin antibodies;
- -Anti-LKM-1 antibodies;
- -Anti- LC1 antibodies; or
- -Anti-SLA/LP antibodies
- Liver Biopsy confirmation of the Diagnosis of auto-immune hepatitis  
This is only covered if the Insured is treated with Immunosuppressive therapy for six (6) months duration or is documented to be under the care of Specialist in gastroenterology or hepatology for six (6) months duration.


--- Chunk 4 ---
## Definition of the Critical Illnesses applicable to: Critical Protector Life (III) on AIA  Guaranteed Protect Plus (III) Accumu